# BERT Models Embeddings Generation
This notebook encodes sentences from a .csv file using multiple BERT embedding models:
- BERT (base)
- RoBERTa (base)
- NeoBERT
- ModernBERT
- AraBERT (base)
- AraBERT (large)
- OpenAI Ada
- OpenAI text-embedding-3 (Large)
- EmbeddingGemma-300m
- Voyage 3 (large)
- BAAI BGE-M3
- e5-large-v2

Embeddings are saved to separate CSV files for each model.

## Setup and Imports


In [26]:
from IPython.display import clear_output

!pip install openai voyageai xformers
!pip install --upgrade transformers torch


In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
from tqdm.auto import tqdm
import warnings
import os
warnings.filterwarnings('ignore')

# Check for CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [49]:
OPENAI_API_KEY = 'YOUR-KEY-HERE'

VOYAGE_API_KEY = 'YOUR-KEY-HERE'

## Load Data


In [30]:
# Load the sentences dataset
df = pd.read_csv('/content/poles_ar.csv')
print(f"Loaded {len(df)} sentences")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Loaded 120 sentences

Columns: ['sentence', 'pos/neg', 'V/A/D']

First few rows:


,sentence,pos/neg,V/A/D
0,الطالب يتبع جميع المتطلبات الأكاديمية بالكامل،...,neg,D
1,أغطية الصوف الناعمة توفر طبقات دافئة مخملية تت...,pos,V
2,العامل يتبع لوائح المنشأة بالكامل، يؤدي المهام...,neg,D
3,العامل يبقى محتجزاً بالكامل في المنشأة، يتبع ج...,neg,D
4,المدير يدير العمليات المؤسسية حصرياً، يحدد الس...,pos,D


## Helper Function for Encoding


In [31]:
def encode_sentences(model_name, sentences, device, batch_size=8):
  print(f"\nLoading {model_name}...")
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModel.from_pretrained(model_name).to(device)
  model.eval()

  embeddings = []

  with torch.no_grad():
      for i in tqdm(range(0, len(sentences), batch_size), desc=f"Encoding with {model_name}"):
          batch = sentences[i:i+batch_size]

          # Tokenize
          inputs = tokenizer(batch, padding=True, truncation=True,
                            max_length=512, return_tensors='pt').to(device)

          # Get embeddings (use [CLS] token)
          outputs = model(**inputs)
          cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
          embeddings.append(cls_embeddings)

  # Clear memory
  del model, tokenizer
  torch.cuda.empty_cache() if torch.cuda.is_available() else None

  return np.vstack(embeddings)


## Helper Function for API-based Models


In [32]:
def encode_sentences_openai(sentences, model_name, batch_size=100):
  try:
      from openai import OpenAI
  except ImportError:
      print("OpenAI library not installed. Install with: pip install openai")
      return None

  client = OpenAI(api_key=OPENAI_API_KEY)
  embeddings = []

  for i in tqdm(range(0, len(sentences), batch_size), desc=f"Encoding with {model_name}"):
      batch = sentences[i:i+batch_size]
      response = client.embeddings.create(
          input=batch,
          model=model_name
      )
      batch_embeddings = [data.embedding for data in response.data]
      embeddings.extend(batch_embeddings)

  return np.array(embeddings)

In [33]:
def encode_sentences_voyage(sentences, model_name, api_key=None, batch_size=128):
  try:
      import voyageai
  except ImportError:
      print("Voyage AI library not installed. Install with: pip install voyageai")
      return None

  vo = voyageai.Client(api_key=VOYAGE_API_KEY)
  embeddings = []

  for i in tqdm(range(0, len(sentences), batch_size), desc=f"Encoding with {model_name}"):
      batch = sentences[i:i+batch_size]
      result = vo.embed(batch, model=model_name)
      embeddings.extend(result.embeddings)

  return np.array(embeddings)

## Helper Function to Save Embeddings


In [34]:
def save_embeddings_to_csv(df, embeddings, model_name, output_path):
  # Create a new dataframe with original columns
  result_df = df.copy()

  # Add embedding dimensions as columns
  embedding_dim = embeddings.shape[1]
  for i in range(embedding_dim):
      result_df[f'emb_{i}'] = embeddings[:, i]

  # Save to CSV
  result_df.to_csv(output_path, index=False)
  print(f"Saved {len(result_df)} rows with {embedding_dim}-dimensional embeddings to {output_path}")


## 1. BERT (base-uncased)


In [35]:
# BERT
bert_embeddings = encode_sentences(
  model_name='bert-base-uncased',
  sentences=df['sentence'].tolist(),
  device=device
)

save_embeddings_to_csv(
  df=df,
  embeddings=bert_embeddings,
  model_name='bert',
  output_path='/content/embeddings/arabic/poles/bert_ar.csv'
)


Loading bert-base-uncased...


Encoding with bert-base-uncased:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 768-dimensional embeddings to /content/embeddings/arabic/poles/bert_ar.csv


## 2. RoBERTa (base)


In [36]:
# RoBERTa
roberta_embeddings = encode_sentences(
    model_name='roberta-base',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=roberta_embeddings,
    model_name='roberta',
    output_path='/content/embeddings/arabic/poles/roberta_ar.csv'
)


Loading roberta-base...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Encoding with roberta-base:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 768-dimensional embeddings to /content/embeddings/arabic/poles/roberta_ar.csv


## 3. NeoBERT


In [37]:
# NeoBERT (using the official model from HuggingFace)
neobert_embeddings = encode_sentences(
    model_name='chandar-lab/NeoBERT',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=neobert_embeddings,
    model_name='neobert',
    output_path='/content/embeddings/arabic/poles/neobert_ar.csv'
)


Loading chandar-lab/NeoBERT...
The repository chandar-lab/NeoBERT contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/chandar-lab/NeoBERT .
 You can inspect the repository content at https://hf.co/chandar-lab/NeoBERT.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Encoding with chandar-lab/NeoBERT:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 768-dimensional embeddings to /content/embeddings/arabic/poles/neobert_ar.csv


## 4. ModernBERT


In [38]:
# ModernBERT (using the base model)
modernbert_embeddings = encode_sentences(
    model_name='answerdotai/ModernBERT-base',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=modernbert_embeddings,
    model_name='modernbert',
    output_path='/content/embeddings/arabic/poles/modernbert_ar.csv'
)


Loading answerdotai/ModernBERT-base...


Encoding with answerdotai/ModernBERT-base:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 768-dimensional embeddings to /content/embeddings/arabic/poles/modernbert_ar.csv


## 5. AraBERT (base)


In [39]:
# AraBERT base
arabert_base_embeddings = encode_sentences(
    model_name='aubmindlab/bert-base-arabertv2',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=arabert_base_embeddings,
    model_name='arabert_base',
    output_path='/content/embeddings/arabic/poles/arabert_base_ar.csv'
)



Loading aubmindlab/bert-base-arabertv2...


Encoding with aubmindlab/bert-base-arabertv2:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 768-dimensional embeddings to /content/embeddings/arabic/poles/arabert_base_ar.csv


## 6. AraBERT (large)


In [40]:
# AraBERT large
arabert_large_embeddings = encode_sentences(
    model_name='aubmindlab/bert-large-arabertv2',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=arabert_large_embeddings,
    model_name='arabert_large',
    output_path='/content/embeddings/arabic/poles/arabert_large_ar.csv'
)



Loading aubmindlab/bert-large-arabertv2...


Encoding with aubmindlab/bert-large-arabertv2:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 1024-dimensional embeddings to /content/embeddings/arabic/poles/arabert_large_ar.csv


## 7. OpenAI Ada


In [41]:
# OpenAI Ada (text-embedding-ada-002)
# Note: Requires OPENAI_API_KEY environment variable to be set
openai_ada_embeddings = encode_sentences_openai(
    sentences=df['sentence'].tolist(),
    model_name='text-embedding-ada-002'
)

if openai_ada_embeddings is not None:
    save_embeddings_to_csv(
        df=df,
        embeddings=openai_ada_embeddings,
        model_name='openai_ada',
        output_path='/content/embeddings/arabic/poles/openai_ada_ar.csv'
    )
else:
    print("Skipping OpenAI Ada")


Encoding with text-embedding-ada-002:   0%|          | 0/2 [00:00<?, ?it/s]

Saved 120 rows with 1536-dimensional embeddings to /content/embeddings/arabic/poles/openai_ada_ar.csv


## 8. OpenAI text-embedding-3-large


In [42]:
# OpenAI text-embedding-3-large
# Note: Requires OPENAI_API_KEY environment variable to be set
openai_3_large_embeddings = encode_sentences_openai(
    sentences=df['sentence'].tolist(),
    model_name='text-embedding-3-large'
)

if openai_3_large_embeddings is not None:
    save_embeddings_to_csv(
        df=df,
        embeddings=openai_3_large_embeddings,
        model_name='openai_3_large',
        output_path='/content/embeddings/arabic/poles/openai_3_large_ar.csv'
    )
else:
    print("Skipping OpenAI text-embedding-3-large - API key not available or library not installed")


Encoding with text-embedding-3-large:   0%|          | 0/2 [00:00<?, ?it/s]

Saved 120 rows with 3072-dimensional embeddings to /content/embeddings/arabic/poles/openai_3_large_ar.csv


## 9. Google EmbeddingGemma-300m


In [43]:
# EmbeddingGemma-300m
# Note: This model may require accepting terms on HuggingFace Hub
!hf auth login

gemma_embeddings = encode_sentences(
    model_name='google/embeddinggemma-300m',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=gemma_embeddings,
    model_name='gemma_300m',
    output_path='/content/embeddings/arabic/poles/gemma_300m_ar.csv'
)



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: read).
The token `Ara

Encoding with google/embeddinggemma-300m:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 768-dimensional embeddings to /content/embeddings/arabic/poles/gemma_300m_ar.csv


## 10. Voyage 3 (Large)


In [44]:
# Voyage 3 Large
# Note: Requires VOYAGE_API_KEY environment variable to be set
voyage_embeddings = encode_sentences_voyage(
    sentences=df['sentence'].tolist(),
    model_name='voyage-3-large'
)

if voyage_embeddings is not None:
    save_embeddings_to_csv(
        df=df,
        embeddings=voyage_embeddings,
        model_name='voyage_3_large',
        output_path='/content/embeddings/arabic/poles/voyage_3_large_ar.csv'
    )
else:
    print("Skipping Voyage 3 Large")


Encoding with voyage-3-large:   0%|          | 0/1 [00:00<?, ?it/s]

Saved 120 rows with 1024-dimensional embeddings to /content/embeddings/arabic/poles/voyage_3_large_ar.csv


## 11. BAAI BGE-M3


In [45]:
# BAAI BGE-M3 (Multilingual model)
bge_m3_embeddings = encode_sentences(
    model_name='BAAI/bge-m3',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=bge_m3_embeddings,
    model_name='bge_m3',
    output_path='/content/embeddings/arabic/poles/bge_m3_ar.csv'
)



Loading BAAI/bge-m3...


Encoding with BAAI/bge-m3:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 1024-dimensional embeddings to /content/embeddings/arabic/poles/bge_m3_ar.csv


## 12. e5-large-v2


In [46]:
# e5-large-v2 (Text Embeddings by Text Representations)
e5_large_embeddings = encode_sentences(
    model_name='intfloat/e5-large-v2',
    sentences=df['sentence'].tolist(),
    device=device
)

save_embeddings_to_csv(
    df=df,
    embeddings=e5_large_embeddings,
    model_name='e5_large_v2',
    output_path='/content/embeddings/arabic/poles/e5_large_v2_ar.csv'
)



Loading intfloat/e5-large-v2...


Encoding with intfloat/e5-large-v2:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 120 rows with 1024-dimensional embeddings to /content/embeddings/arabic/poles/e5_large_v2_ar.csv


## Summary: Embedding Dimensions


In [47]:
print("=" * 60)
print("EMBEDDING DIMENSIONS SUMMARY")
print("=" * 60)

# HuggingFace models
print(f"  1. BERT (base):              {bert_embeddings.shape[1]} dims")
print(f"  2. RoBERTa (base):           {roberta_embeddings.shape[1]} dims")
print(f"  3. NeoBERT:                  {neobert_embeddings.shape[1]} dims")
print(f"  4. ModernBERT:               {modernbert_embeddings.shape[1]} dims")
print(f"  5. AraBERT (base):           {arabert_base_embeddings.shape[1]} dims")
print(f"  6. AraBERT (large):          {arabert_large_embeddings.shape[1]} dims")

# API-based models
if openai_ada_embeddings is not None:
    print(f"  7. OpenAI Ada:               {openai_ada_embeddings.shape[1]} dims")
else:
    print(f"  7. OpenAI Ada:               [Not available]")

if openai_3_large_embeddings is not None:
    print(f"  8. OpenAI 3-large:           {openai_3_large_embeddings.shape[1]} dims")
else:
    print(f"  8. OpenAI 3-large:           [Not available]")

print(f"  9. EmbeddingGemma-300m:      {gemma_embeddings.shape[1]} dims")

if voyage_embeddings is not None:
    print(f" 10. Voyage 3 (Large):         {voyage_embeddings.shape[1]} dims")
else:
    print(f" 10. Voyage 3 (Large):         [Not available]")

print(f" 11. BAAI BGE-M3:              {bge_m3_embeddings.shape[1]} dims")
print(f" 12. e5-large-v2:              {e5_large_embeddings.shape[1]} dims")
print("=" * 60)


EMBEDDING DIMENSIONS SUMMARY
  1. BERT (base):              768 dims
  2. RoBERTa (base):           768 dims
  3. NeoBERT:                  768 dims
  4. ModernBERT:               768 dims
  5. AraBERT (base):           768 dims
  6. AraBERT (large):          1024 dims
  7. OpenAI Ada:               1536 dims
  8. OpenAI 3-large:           3072 dims
  9. EmbeddingGemma-300m:      768 dims
 10. Voyage 3 (Large):         1024 dims
 11. BAAI BGE-M3:              1024 dims
 12. e5-large-v2:              1024 dims
